# compare_versions.ipynb

Scans all `*/results/*.json` files, loads the most recent run per
`(version, data_source)` pair, and prints a side-by-side comparison table
with deltas vs the v2/backtest baseline.

Re-run at any time as new results accumulate.


In [1]:
import json, sys
from pathlib import Path
import pandas as pd

HERE   = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
V2_BT  = HERE.parent / 'v2' / 'backtest' / 'results'
V4_DIR = HERE

# ── Scan all results/ folders ─────────────────────────────────────────────────
def load_latest(root):
    rows = []
    for json_file in sorted(root.rglob('*.json')):
        if 'trade_cache' in json_file.name: continue
        try:
            with open(json_file, encoding='utf-8') as f:
                d = json.load(f)
            row = {'version': d.get('version',''), 'key_change': d.get('key_change',''),
                   'data_source': d.get('data_source',''), 'run_id': d.get('run_id',''),
                   '_file': json_file}
            row.update(d.get('results', {}))
            rows.append(row)
        except Exception as e:
            print(f"  [warn] {json_file.name}: {e}")
    return rows

all_rows = load_latest(V2_BT) + load_latest(V4_DIR)
df_all   = pd.DataFrame(all_rows)
print(f"Found {len(df_all)} result files across all versions.")
df_all[['version','data_source','run_id']].head(20)


Found 8 result files across all versions.


,version,data_source,run_id
0,v2_backtest,real_2024_options,v2_backtest_20260417_0212_real2024options
1,v4.1,real_2024_options,v4.1_20260417_0212_real2024options
2,v4.2,real_2024_options,v4.2_20260417_0212_real2024options
3,v4.3,real_2024_options,v4.3_20260417_0212_real2024options
4,v4.4,real_2024_options,v4.4_20260417_0215_real2024options
5,v4.5,real_2024_options,v4.5_20260417_0216_real2024options
6,v4.6,real_2024_options,v4.6_20260417_0214_real2024options
7,v4.7,real_2024_options,v4.7_20260417_0215_real2024options


In [2]:
# ── Keep most recent run per (version, data_source) ───────────────────────────
if df_all.empty:
    print("No results yet — run some backtest notebooks first.")
else:
    df_latest = (df_all
                 .sort_values('run_id', ascending=False)
                 .drop_duplicates(subset=['version','data_source'])
                 .sort_values(['data_source','xirr_pct'], ascending=[True,False])
                 .reset_index(drop=True))

    COLS = ['version','data_source','total_trades','profit_trade_pct',
            'tp_hit_pct','sl_hit_pct','time_exit_pct',
            'xirr_pct','max_drawdown_pct','net_return_pct','key_change']
    avail = [c for c in COLS if c in df_latest.columns]
    print("\n=== SUMMARY TABLE (sorted by XIRR) ===")
    display(df_latest[avail])



=== SUMMARY TABLE (sorted by XIRR) ===


,version,data_source,total_trades,profit_trade_pct,tp_hit_pct,sl_hit_pct,time_exit_pct,xirr_pct,max_drawdown_pct,net_return_pct,key_change
0,v4.3,real_2024_options,60,36.67,33.33,61.67,5.00,413.39,40.3186,393.61,SGX signal: ^N225 trade-day open (Tokyo 05:30 ...
1,v2_backtest,real_2024_options,57,36.84,33.33,61.40,5.26,347.99,42.0718,1597.79,"Baseline — v2 signals, 2024 real options data,..."
2,v4.2,real_2024_options,75,34.67,32.00,61.33,6.67,296.15,62.4093,154.53,Include Monday sessions (Friday US close used ...
3,v4.5,real_2024_options,62,35.48,32.26,62.90,4.84,241.61,54.8824,231.42,Signal combos rebuilt on dir_120 (11:15 AM out...
4,v4.4,real_2024_options,55,36.36,32.73,61.82,5.45,203.24,43.1590,177.01,India VIX regime filter: skip if D-1 ^INDIAVIX...
5,v4.1,real_2024_options,56,35.71,32.14,60.71,7.14,148.84,47.4209,382.80,"SGX proxy: NKD=F → ^N225 (Nikkei 225 cash, JPY..."
6,v4.6,real_2024_options,6,50.00,33.33,50.00,16.67,70.52,16.7074,28.64,NASDAQ confirmation filter: only trade if NASD...
7,v4.7,real_2024_options,22,31.82,31.82,63.64,4.55,15.60,57.5668,13.82,9:15-9:25 direction filter: only enter if NIFT...


In [3]:
# ── Delta table vs v2/backtest baseline ───────────────────────────────────────
if not df_all.empty:
    for ds in df_latest['data_source'].unique():
        sub = df_latest[df_latest['data_source'] == ds].copy()
        baseline_rows = sub[sub['version'].str.replace('_bs', '', regex=False) == 'v2_backtest']
        if baseline_rows.empty:
            print(f"  [{ds}] No v2/backtest baseline yet — skipping delta.")
            continue
        baseline = baseline_rows.iloc[0]

        DELTA_COLS = ['total_trades','profit_trade_pct','xirr_pct',
                      'max_drawdown_pct','net_return_pct']
        delta_rows = []
        for _, r in sub[sub['version'] != 'v2_backtest'].iterrows():
            row = {'version': r['version'], 'key_change': r.get('key_change','')}
            for c in DELTA_COLS:
                try:
                    row[f'Δ {c}'] = round(float(r[c]) - float(baseline[c]), 2)
                except Exception:
                    row[f'Δ {c}'] = None
            delta_rows.append(row)

        if delta_rows:
            print(f"\n=== DELTA vs v2/backtest BASELINE  [{ds}] ===")
            display(pd.DataFrame(delta_rows))



=== DELTA vs v2/backtest BASELINE  [real_2024_options] ===


,version,key_change,Δ total_trades,Δ profit_trade_pct,Δ xirr_pct,Δ max_drawdown_pct,Δ net_return_pct
0,v4.3,SGX signal: ^N225 trade-day open (Tokyo 05:30 ...,3.0,-0.17,65.40,-1.75,-1204.18
1,v4.2,Include Monday sessions (Friday US close used ...,18.0,-2.17,-51.84,20.34,-1443.26
2,v4.5,Signal combos rebuilt on dir_120 (11:15 AM out...,5.0,-1.36,-106.38,12.81,-1366.37
3,v4.4,India VIX regime filter: skip if D-1 ^INDIAVIX...,-2.0,-0.48,-144.75,1.09,-1420.78
4,v4.1,"SGX proxy: NKD=F → ^N225 (Nikkei 225 cash, JPY...",-1.0,-1.13,-199.15,5.35,-1214.99
5,v4.6,NASDAQ confirmation filter: only trade if NASD...,-51.0,13.16,-277.47,-25.36,-1569.15
6,v4.7,9:15-9:25 direction filter: only enter if NIFT...,-35.0,-5.02,-332.39,15.49,-1583.97
